# 11 - Figures and tables

**CPU fine. Run all; idempotent.** Every figure and table regenerates from
saved artifacts (manifest, predictions, SHAP values, result CSVs) in one pass -
nothing is hand-assembled, so nothing can drift from the numbers in the text.

Outputs: `results/figures/*.pdf` (vector, for the manuscript) and `.png`
(300 dpi, for drafts), plus `results/tables/*.tex`.

In [ ]:
# --- standard header ---
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, getpass
REPO = '/content/secure-dns-trust-ai'
URL  = 'github.com/sandesh20lamichhane/secure-dns-trust-ai.git'
if os.path.isdir(REPO):
    subprocess.run(['git','-C',REPO,'pull','-q'], check=False)
else:
    TOKEN = getpass.getpass('GitHub PAT: ')
    subprocess.run(['git','clone','-q',f'https://{TOKEN}@{URL}',REPO], check=True)
sys.path.insert(0, REPO)
os.environ['DNSTRUST_CONFIG_DIR'] = f'{REPO}/configs'

from src.utils import config, manifest, seeds
P = config.paths(); config.ensure_tree(P); seeds.set_all(42)
print('repo', manifest.git_sha(REPO))

In [ ]:
!pip -q install pyarrow zstandard matplotlib

In [ ]:
import pandas as pd, numpy as np, matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
from src.utils import manifest as mf
from src.evaluate import predictions

TAB, FIG = Path(P['results']['tables']), Path(P['results']['figures'])
FIG.mkdir(parents=True, exist_ok=True)
PRED_DIR = Path(P['artifacts']['predictions']); SHAP_DIR = Path(P['artifacts']['shap_values'])

matplotlib.rcParams.update({'font.size': 9, 'font.family': 'serif',
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.dpi': 110, 'savefig.dpi': 300, 'savefig.bbox': 'tight'})
FD, RD = 'family_disjoint_v1', 'random_v1'
C_FD, C_RD, C_ACC = '#B3432B', '#3B6B8F', '#6B7B4C'

def save(fig, name):
    fig.savefig(FIG/f'{name}.pdf'); fig.savefig(FIG/f'{name}.png'); plt.close(fig)
    print('wrote', name)

from pathlib import Path as _P
assert _P(P['manifest']).exists(), (
    f"manifest not found at {P['manifest']} - files present: "
    f"{[f.name for f in _P(P['manifest']).parent.glob('run_manifest*')]}")
man = mf.load_manifest(P['manifest'])
n_raw = len(man)
assert n_raw > 0 and 'timestamp' in man.columns, 'manifest is empty or malformed'
# Re-running a notebook appends a fresh row per run_id; keep only the latest so
# no table averages over superseded runs (this once contaminated the trust-score row).
man = man.sort_values('timestamp').groupby('run_id').tail(1).reset_index(drop=True)
print('manifest rows:', n_raw, '-> unique runs:', len(man))

## Fig 1 - certificate coverage by class and source (the fusion motivation)

In [ ]:
cov = pd.DataFrame({
    'source': ['Tranco\n(benign)', 'URLhaus\n(malware)', 'OpenPhish\n(phishing)', 'UMUDGA\n(DGA)'],
    'rate': [0.833, 0.800, 0.566, 0.006]})
fig, ax = plt.subplots(figsize=(3.4, 2.4))
bars = ax.bar(cov['source'], cov['rate']*100,
              color=[C_RD, C_FD, C_FD, C_FD], width=0.62)
for b, v in zip(bars, cov['rate']):
    ax.text(b.get_x()+b.get_width()/2, v*100+1.5, f'{v*100:.1f}%', ha='center', fontsize=8)
ax.set_ylabel('domains serving TLS (%)'); ax.set_ylim(0, 100)
save(fig, 'fig1_certificate_coverage')

## Fig 2 - random-split optimism (tier A, four models)

In [ ]:
fams = {'xgb_tierA_lexical': 'XGBoost\n(lexical+tld)', 'xgb_tierA_notld': 'XGBoost\n(lexical, no tld)',
        'cnnbilstm_tierA': 'CNN-BiLSTM', 'cnnbilstm_balanced_tierA': 'CNN-BiLSTM\n(balanced)'}
a = man[man['run_family'].isin(fams)]
roc = a.groupby(['run_family','split_name'])['metrics.roc_auc'].agg(['mean','std']).unstack('split_name')
order = list(fams)
x = np.arange(len(order)); w = 0.36
fig, ax = plt.subplots(figsize=(4.6, 2.6))
ax.bar(x-w/2, [roc.loc[f,('mean',RD)] for f in order], w, label='random split',
       color=C_RD, yerr=[roc.loc[f,('std',RD)] for f in order], capsize=2)
ax.bar(x+w/2, [roc.loc[f,('mean',FD)] for f in order], w, label='family-disjoint',
       color=C_FD, yerr=[roc.loc[f,('std',FD)] for f in order], capsize=2)
ax.set_xticks(x, [fams[f] for f in order], fontsize=8)
ax.set_ylabel('ROC-AUC'); ax.set_ylim(0.85, 1.005); ax.legend(frameon=False, fontsize=8)
save(fig, 'fig2_random_split_optimism')

## Fig 3 - fusion main result (FPR@95%TPR by component, family-disjoint)

In [ ]:
f = man[man['run_family'].str.startswith('fusion_')].copy()
f['row'] = f['run_family'].str.replace('fusion_','')
labels = {'a_lexical':'lexical\nonly','b_cert':'certificate\nonly','e_late':'late fusion\n(calibrated\nmean)',
          'd_fused_emb':'fused +\nCNN\nembedding','c_fused':'fused\n(trust\nscore)'}
order = ['a_lexical','b_cert','e_late','d_fused_emb','c_fused']
g = f[f.split_name==FD].groupby('row')['metrics.fpr_at_95_tpr'].agg(['mean','std'])
fig, ax = plt.subplots(figsize=(4.6, 2.8))
cols = ['#9AA5B1']*4 + [C_ACC]
bars = ax.bar(range(len(order)), [g.loc[r,'mean']*100 for r in order],
              yerr=[g.loc[r,'std']*100 for r in order], capsize=2,
              color=[cols[i] for i in range(len(order))], width=0.62)
for i, r in enumerate(order):
    ax.text(i, g.loc[r,'mean']*100+1.2, f"{g.loc[r,'mean']*100:.1f}%", ha='center', fontsize=8)
ax.set_xticks(range(len(order)), [labels[r] for r in order], fontsize=7.5)
ax.set_ylabel('FPR @ 95% TPR (%)  [lower is better]')
ax.set_title('Family-disjoint evaluation', fontsize=9)
save(fig, 'fig3_fusion_main_result')

## Fig 4 - reliability diagrams (raw vs hybrid-calibrated, per regime)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(7.4, 2.5), sharey=True)
for ax, reg, title in zip(axes, ['all','nocert','cert'],
                          ['all domains','no-certificate regime','certificate-holder regime']):
    for name, color in [('raw','#9AA5B1'), ('calibrated', C_ACC)]:
        c = pd.read_csv(TAB/f'reliability_{FD}_{name}_{reg}.csv')
        ax.plot(c['mean_predicted'], c['observed_frequency'], 'o-', ms=3, lw=1,
                color=color, label=name)
    ax.plot([0,1],[0,1],'--',lw=0.8,color='#444')
    ax.set_title(title, fontsize=8.5); ax.set_xlabel('predicted probability')
axes[0].set_ylabel('observed frequency'); axes[0].legend(frameon=False, fontsize=8)
save(fig, 'fig4_reliability')

## Fig 5 - deferral: zones and the operator's knob

In [ ]:
op = pd.read_csv(TAB/'table_operating_points.csv')
sw = pd.read_csv(TAB/'table_deferral_sweep.csv')
fig, axes = plt.subplots(1, 2, figsize=(6.8, 2.6))

z = op[op.split==FD][['pass_frac','defer_frac','block_frac']].mean()
axes[0].barh(['policy'], [z['pass_frac']*100], color=C_RD, label='pass')
axes[0].barh(['policy'], [z['defer_frac']*100], left=[z['pass_frac']*100], color='#D9B44A',
             label='defer to DANE/TLSA')
axes[0].barh(['policy'], [z['block_frac']*100], left=[(z['pass_frac']+z['defer_frac'])*100],
             color=C_FD, label='block pending validation')
for frac, xoff in [(z['pass_frac'], z['pass_frac']/2),
                   (z['defer_frac'], z['pass_frac']+z['defer_frac']/2),
                   (z['block_frac'], z['pass_frac']+z['defer_frac']+z['block_frac']/2)]:
    axes[0].text(xoff*100, 0, f'{frac*100:.0f}%', ha='center', va='center', fontsize=8)
axes[0].set_xlim(0,100); axes[0].set_yticks([]); axes[0].set_xlabel('share of traffic (%)')
axes[0].legend(frameon=False, fontsize=7, ncol=1, loc='upper center', bbox_to_anchor=(0.5,-0.35))
axes[0].set_title('Zones (family-disjoint, FPR target 0.1%)', fontsize=8.5)

for split, color in [(FD, C_FD), (RD, C_RD)]:
    d = sw[sw.split==split]
    axes[1].plot(d['block_benign_rate']*100, d['defer_frac']*100, 'o-', color=color,
                 label=split.replace('_v1','').replace('_',' '))
    for _, r in d.iterrows():
        axes[1].annotate(f"{r['target_fpr']*100:.1f}%", (r['block_benign_rate']*100, r['defer_frac']*100),
                         textcoords='offset points', xytext=(4,3), fontsize=6.5)
axes[1].set_xlabel('benign collateral in block zone (%)')
axes[1].set_ylabel('deferral fraction (%)'); axes[1].legend(frameon=False, fontsize=7)
axes[1].set_title("Operator's knob: FPR target", fontsize=8.5)
save(fig, 'fig5_deferral')

## Fig 6 - adaptivity: per-regime attribution mass and top features

In [ ]:
reg = pd.read_csv(TAB/'table_shap_per_regime.csv')
fd = reg[reg['split']==FD].set_index('feature') if 'split' in reg.columns else reg.set_index('feature')
LEX = ['length','core_length','n_labels','shannon_entropy','vowel_ratio','digit_ratio','hyphen_count',
       'max_consec_consonants','bigram_score','trigram_score','unique_char_ratio','is_idn',
       'has_digit','starts_with_digit']
mass = {}
for r in ('nocert','cert'):
    tot = fd[r].sum()
    mass[r] = (fd.loc[[f for f in LEX if f in fd.index], r].sum()/tot,
               fd.loc[[f for f in fd.index if f not in LEX], r].sum()/tot)
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.0), gridspec_kw={'width_ratios':[1,1.7], 'wspace':0.55})
x = [0,1]
axes[0].bar(x, [mass['nocert'][0]*100, mass['cert'][0]*100], 0.55, label='lexical features', color=C_RD)
axes[0].bar(x, [mass['nocert'][1]*100, mass['cert'][1]*100], 0.55,
            bottom=[mass['nocert'][0]*100, mass['cert'][0]*100], label='certificate features', color=C_FD)
axes[0].set_xticks(x, ['no-certificate\nregime','certificate-holder\nregime'], fontsize=8)
for i, r in enumerate(('nocert','cert')):
    axes[0].text(i, mass[r][0]*50, f'{mass[r][0]*100:.0f}%', ha='center', va='center', color='white', fontsize=8)
    axes[0].text(i, mass[r][0]*100 + mass[r][1]*50, f'{mass[r][1]*100:.0f}%', ha='center', va='center', color='white', fontsize=8)
axes[0].set_ylabel('attribution mass (%)'); axes[0].set_ylim(0, 105)
axes[0].legend(frameon=False, fontsize=7, loc='lower center', bbox_to_anchor=(0.5, 1.02), handlelength=1)
axes[0].text(-0.55, 112, '(a)', fontsize=9, fontweight='bold')

top = fd.assign(t=fd['nocert']+fd['cert']).sort_values('t', ascending=True).tail(10)
y = np.arange(len(top))
axes[1].barh(y-0.19, top['nocert'], 0.38, color=C_RD, label='no-certificate')
axes[1].barh(y+0.19, top['cert'], 0.38, color=C_FD, label='certificate-holder')
axes[1].set_yticks(y, top.index, fontsize=7.5)
axes[1].set_xlabel('mean |SHAP|'); axes[1].legend(frameon=False, fontsize=7, loc='lower right')
axes[1].text(-0.95, len(top)-0.2, '(b)', fontsize=9, fontweight='bold')
save(fig, 'fig6_adaptivity_attribution')

## Fig 7 - case studies (per-domain explanations)

In [ ]:
S = pd.read_parquet(SHAP_DIR/f'fusion_c_fused_{FD}_s42.parquet')
ts = predictions.load(f'trustscore_{FD}_s42', PRED_DIR).set_index('domain')
S = S.set_index('domain'); FEATS = [c for c in S.columns if c not in ('_bias','label','has_certificate')]
cases = ['aybjifmmfwluykcxe.com', 'dreamlike.art', 'ltpqsnu.biz']
titles = ['DGA, unseen family\n(blocked)', 'benign, no certificate\n(deferred -> DANE decides)',
          'top-ranked certificate-holder\n(passes; regime posterior is small)']
fig, axes = plt.subplots(1, 3, figsize=(7.6, 2.6), sharex=False)
for ax, dom, title in zip(axes, cases, titles):
    if dom not in S.index: ax.set_visible(False); continue
    contrib = S.loc[dom, FEATS].astype(float)
    top = contrib.reindex(contrib.abs().sort_values(ascending=False).index[:6])[::-1]
    ax.barh(range(len(top)), top.values, color=[C_FD if v>0 else C_RD for v in top.values], height=0.6)
    ax.set_yticks(range(len(top)), top.index, fontsize=7)
    ax.axvline(0, color='#444', lw=0.8)
    sc = ts.loc[dom, 'calibrated_score'] if dom in ts.index else float('nan')
    ax.set_title(f'{dom}\nscore {sc:.3f} - {title}', fontsize=7.5)
    ax.set_xlabel('SHAP contribution', fontsize=7.5)
save(fig, 'fig7_case_studies')

## Headline table - every model, both splits, mean +/- std

In [ ]:
fams = {'xgb_tierA_lexical':'Tier A: XGB lexical+tld','xgb_tierA_notld':'Tier A: XGB lexical (no tld)',
 'cnnbilstm_tierA':'Tier A: CNN-BiLSTM','cnnbilstm_balanced_tierA':'Tier A: CNN-BiLSTM balanced',
 'fusion_a_lexical':'Probe: lexical only','fusion_b_cert':'Probe: certificate only',
 'fusion_e_late':'Probe: late fusion','fusion_d_fused_emb':'Probe: fused + embedding',
 'fusion_c_fused':'Probe: FUSED TRUST SCORE','trustscore':'Trust score (hybrid-calibrated)'}
a = man[man['run_family'].isin(fams)].copy()
a['Model'] = a['run_family'].map(fams)
def fmt(g):
    return pd.Series({
      'ROC-AUC': f"{g['metrics.roc_auc'].mean():.4f} ± {g['metrics.roc_auc'].std():.4f}",
      'FPR@95%TPR': f"{g['metrics.fpr_at_95_tpr'].mean():.4f} ± {g['metrics.fpr_at_95_tpr'].std():.4f}",
      'ECE': f"{g['metrics.ece'].mean():.4f}" if 'metrics.ece' in g else ''})
head = a.groupby(['Model','split_name']).apply(fmt, include_groups=False).unstack('split_name')
display(head)
head.to_csv(TAB/'table_headline.csv')
with open(TAB/'table_headline.tex','w') as fh: fh.write(head.to_latex())
print('wrote headline table (csv + tex)')

In [ ]:
print('FIGURES:'); [print(' ', f.name) for f in sorted(FIG.glob('*.pdf'))]
print('TABLES:');  [print(' ', f.name) for f in sorted(TAB.glob('*.csv'))]

---
Everything above regenerates from artifacts alone. Next: the manuscript.